# Building an A2A-Compliant Agent from a LangGraph Graph

This is notebook **2 of 3** in the `09_Agent_Protocols/A2A/` track:

1. `01_Foundations/01_A2A_Protocol_Basics.ipynb` — raw protocol mechanics (Agent Card, Task, Message, a minimal echo server) using the `a2a-sdk` directly.
2. **This notebook** — wraps a *real, LLM-powered* LangGraph agent behind the A2A protocol, so it becomes callable by any A2A-compliant client. This is the actual point of "building A2A-compliant agents" — a step up from a toy echo server.
3. `03_Applications/` — a cross-framework interop demo (authored separately).

## What we are going to do

1. Build a small, genuinely useful **LangGraph ReAct agent** with two tools — a `calculator` and a `lookup` over an in-notebook mini knowledge base — using `from helpers import get_llm` (this repo's LLM factory convention; never a direct provider client).
2. Wrap the LangGraph agent's `.invoke()` inside an **A2A `AgentExecutor`**, so an incoming A2A task/message gets routed to the LangGraph agent and its final answer becomes the A2A response.
3. Serve a proper **`AgentCard`** describing the agent's real capabilities (name, description, skills: `calculator`, `lookup`) at `/.well-known/agent.json`.
4. Run the A2A server in a background thread, then act as an **A2A client** — using `A2ACardResolver` + `A2AClient`, the same pattern as this repo's `mcp_a2a_agentic_rag/utilities/a2a/agent_discovery.py` and `agent_connect.py` — to discover the card and send a task that forces the agent to use its tools.
5. Print the full A2A response and discuss it.

We deliberately mirror the API conventions already used in this repo's applied build at `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/` (`AgentExecutor`, `AgentCard`/`AgentSkill`/`AgentCapabilities`, `DefaultRequestHandler`, `InMemoryTaskStore`, `A2AStarletteApplication`, `A2ACardResolver`, `A2AClient`) rather than inventing new patterns.

## Setup

Dependencies: `langgraph`, `langchain-core`, `a2a-sdk`, `uvicorn`, `httpx`, `nest_asyncio` (to run an asyncio server thread cleanly from inside a running notebook event loop).

In [ ]:
# ============ INSTALL (uncomment if needed) ============
# %pip install -q langgraph langchain-core a2a-sdk uvicorn httpx nest_asyncio

In [ ]:
# ============ IMPORTS ============
import asyncio
import threading
import time
from uuid import uuid4

import httpx
import nest_asyncio
import uvicorn

# LangGraph side
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# This repo's LLM factory convention -- never instantiate a provider client directly
from helpers import get_llm

# A2A server side (mirrors agents/agentic_rag_agent/{agent_executor,main}.py
# in 09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    Task,
    TaskState,
    UnsupportedOperationError,
)
from a2a.utils import new_agent_text_message, new_task
from a2a.utils.errors import ServerError

# A2A client side (mirrors utilities/a2a/agent_connect.py and agent_discovery.py
# in the same mcp_a2a_agentic_rag build)
from a2a.client import A2ACardResolver, A2AClient
from a2a.types import MessageSendParams, SendMessageRequest

nest_asyncio.apply()

HOST = "localhost"
PORT = 10101
BASE_URL = f"http://{HOST}:{PORT}"

## Step 1 — Build a real LangGraph ReAct agent

The agent gets two tools:

- `calculator` — evaluates a basic arithmetic expression.
- `lookup_company_fact` — looks up a fact about a fictional company from a small in-notebook dictionary (stands in for any real data source / retriever).

This is exactly the kind of agent built throughout `05_AI_Agent_Fundamentals/` — the only new thing in this notebook is putting an A2A server *in front of* it.

In [ ]:
# ============ IN-NOTEBOOK MINI DATASET ============
COMPANY_FACTS = {
    "contoso": {"founded": 1998, "industry": "cloud software", "hq": "Seattle, WA"},
    "fabrikam": {"founded": 2005, "industry": "consumer electronics", "hq": "Austin, TX"},
    "northwind": {"founded": 1987, "industry": "logistics", "hq": "Rotterdam, NL"},
}


# ============ TOOLS ============
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '234 * 56' or '(12 + 8) / 4'."""
    allowed_chars = set("0123456789+-*/(). ")
    if not set(expression) <= allowed_chars:
        return "Error: expression contains unsupported characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:  # noqa: BLE001
        return f"Error evaluating expression: {exc}"


@tool
def lookup_company_fact(company_name: str) -> str:
    """Look up founding year, industry, and HQ for a known fictional company
    (contoso, fabrikam, northwind)."""
    facts = COMPANY_FACTS.get(company_name.strip().lower())
    if not facts:
        return f"No record found for '{company_name}'. Known companies: {list(COMPANY_FACTS)}"
    return (
        f"{company_name.title()} was founded in {facts['founded']}, "
        f"operates in {facts['industry']}, and is headquartered in {facts['hq']}."
    )


# ============ LANGGRAPH REACT AGENT ============
llm = get_llm(temperature=0)
langgraph_agent = create_react_agent(llm, tools=[calculator, lookup_company_fact])

In [ ]:
# ============ SANITY-CHECK THE AGENT LOCALLY (BEFORE WRAPPING IN A2A) ============
local_result = langgraph_agent.invoke(
    {"messages": [("user", "What is 234 * 56, and when was Contoso founded?")]}
)
print(local_result["messages"][-1].content)

### Discussion of the output

The graph correctly routes the arithmetic sub-question to `calculator` and the company sub-question to `lookup_company_fact`, then composes both tool results into a single final answer. This `.invoke()` call is exactly what we will call from inside the A2A `AgentExecutor` below — the A2A layer never needs to know the agent is built with LangGraph.

## Step 2 — Wrap the LangGraph agent in an A2A `AgentExecutor`

`AgentExecutor.execute()` is the seam between the A2A protocol and any underlying agent implementation. Here we:

1. Pull the user's text out of `RequestContext.get_user_input()`.
2. Create the A2A `Task` if one doesn't already exist for this exchange.
3. Emit a `TaskState.working` status update (so a streaming client sees progress).
4. Run the LangGraph agent's `.invoke()` in a worker thread (it's a synchronous call) and take its final message as the answer.
5. Emit a `TaskState.completed` status update carrying that answer as the final `Message`.

This mirrors `agents/agentic_rag_agent/agent_executor.py` in `09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/`, simplified to a single-shot (non-streaming-from-the-agent) call since LangGraph's `.invoke()` returns once, synchronously.

In [ ]:
# ============ A2A AGENT EXECUTOR WRAPPING THE LANGGRAPH AGENT ============
class LangGraphAgentExecutor(AgentExecutor):
    """Adapts a compiled LangGraph graph to the A2A AgentExecutor interface."""

    def __init__(self, graph):
        self.graph = graph

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()
        task = context.current_task

        if not task:
            task = new_task(context.message)
            await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)

        try:
            await updater.update_status(
                TaskState.working,
                new_agent_text_message(
                    "LangGraph agent is reasoning over your request...",
                    task.context_id,
                    task.id,
                ),
            )

            # LangGraph's .invoke() is synchronous -- run it off the event loop
            # so the A2A server stays responsive.
            result = await asyncio.to_thread(
                self.graph.invoke,
                {"messages": [("user", user_input)]},
            )
            final_answer = result["messages"][-1].content

            await updater.update_status(
                TaskState.completed,
                new_agent_text_message(final_answer, task.context_id, task.id),
            )
        except Exception as exc:  # noqa: BLE001
            await updater.update_status(
                TaskState.failed,
                new_agent_text_message(f"LangGraph agent error: {exc}", task.context_id, task.id),
            )
            raise

    async def cancel(self, request: RequestContext, event_queue: EventQueue) -> Task | None:
        """Cancellation is not supported for this minimal teaching agent."""
        raise ServerError(error=UnsupportedOperationError())

## Step 3 — Describe the agent with a real `AgentCard`

The `AgentCard` is what makes this agent *discoverable*: any A2A client can fetch it from `/.well-known/agent.json` and learn what the agent can do, without any framework-specific knowledge of LangGraph. We declare two `AgentSkill`s that map directly onto the two real tools the LangGraph agent has.

In [ ]:
# ============ AGENT CARD (served at /.well-known/agent.json) ============
calculator_skill = AgentSkill(
    id="calculator",
    name="calculator",
    description="Evaluates basic arithmetic expressions.",
    tags=["math", "arithmetic"],
    examples=["What is 234 * 56?", "Compute (12 + 8) / 4"],
)

lookup_skill = AgentSkill(
    id="lookup",
    name="lookup",
    description="Looks up founding year, industry, and HQ for a small set of known companies.",
    tags=["lookup", "knowledge-base"],
    examples=["When was Contoso founded?", "Where is Fabrikam headquartered?"],
)

agent_card = AgentCard(
    name="langgraph_calculator_lookup_agent",
    description=(
        "A LangGraph ReAct agent, served over A2A, that can do arithmetic "
        "and look up facts about a small set of known companies."
    ),
    url=f"{BASE_URL}/",
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    skills=[calculator_skill, lookup_skill],
    capabilities=AgentCapabilities(streaming=True),
)

agent_card.model_dump(exclude_none=True)

## Step 4 — Assemble and run the A2A server

Same assembly as `agents/agentic_rag_agent/main.py`: `DefaultRequestHandler` + `InMemoryTaskStore` + `A2AStarletteApplication`, served with `uvicorn`. Since we're inside a notebook (which already owns an event loop), we run the server in a background thread with its own event loop rather than calling `asyncio.run()` directly.

In [ ]:
# ============ ASSEMBLE THE A2A SERVER APP ============
request_handler = DefaultRequestHandler(
    agent_executor=LangGraphAgentExecutor(langgraph_agent),
    task_store=InMemoryTaskStore(),
)

server = A2AStarletteApplication(agent_card=agent_card, http_handler=request_handler)
app = server.build()

uvicorn_config = uvicorn.Config(app, host=HOST, port=PORT, log_level="warning")
uvicorn_server = uvicorn.Server(uvicorn_config)

In [ ]:
# ============ RUN THE SERVER IN A BACKGROUND THREAD ============
def _run_server():
    asyncio.run(uvicorn_server.serve())


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Give uvicorn a moment to bind the port before the client tries to connect.
time.sleep(1.5)
print(f"A2A server for '{agent_card.name}' listening at {BASE_URL}")

## Step 5 — Act as an A2A client: discover the card, then send a task

This mirrors `utilities/a2a/agent_discovery.py` (`A2ACardResolver.get_agent_card()`) and `utilities/a2a/agent_connect.py` (`A2AClient` over an `httpx.AsyncClient`) from `mcp_a2a_agentic_rag`. We ask a question that requires **both** tools, so the response can only be correct if the LangGraph agent actually reasoned and called them.

In [ ]:
# ============ DISCOVER THE AGENT CARD ============
async def discover_agent_card(base_url: str):
    async with httpx.AsyncClient(timeout=30.0) as httpx_client:
        resolver = A2ACardResolver(base_url=base_url.rstrip("/"), httpx_client=httpx_client)
        return await resolver.get_agent_card()


discovered_card = asyncio.run(discover_agent_card(BASE_URL))
print(f"Discovered agent: {discovered_card.name}")
print(f"Description: {discovered_card.description}")
print(f"Skills: {[skill.id for skill in discovered_card.skills]}")

In [ ]:
# ============ SEND A TASK THAT REQUIRES BOTH TOOLS ============
async def send_task(base_url: str, message_text: str) -> dict:
    async with httpx.AsyncClient(timeout=300.0) as httpx_client:
        client = A2AClient(httpx_client=httpx_client, url=base_url.rstrip("/"))

        request = SendMessageRequest(
            id=str(uuid4()),
            params=MessageSendParams(
                message={
                    "role": "user",
                    "parts": [{"kind": "text", "text": message_text}],
                    "message_id": str(uuid4()),
                }
            ),
        )

        response = await client.send_message(request)
        return response.model_dump(mode="json", exclude_none=True)


query = "What is 234 * 56, and when was Contoso founded?"
full_response = asyncio.run(send_task(BASE_URL, query))

import json

print(json.dumps(full_response, indent=2, ensure_ascii=False))

### Discussion of the output

The client never imports LangGraph, and never calls `.invoke()` directly — it only speaks A2A: it resolved the `AgentCard` from `/.well-known/agent.json`, then sent a `Message` and got back a `Task` (or terminal `Message`) whose final text carries the same answer we saw when we called the LangGraph agent locally in Step 1. Any other A2A-compliant client (a different framework, a different language, another agent in a multi-agent system) could have made this exact call with no knowledge of what's running behind the URL — that decoupling is the entire point of the protocol.

In [ ]:
# ============ CLEANUP: STOP THE SERVER ============
uvicorn_server.should_exit = True
server_thread.join(timeout=5)
print("A2A server stopped.")

## This is the same pattern as the repo's applied A2A build, at smaller scale

`09_Agent_Protocols/MCP/mcp_a2a_agentic_rag/` does exactly this — an `AgentExecutor` (`agents/agentic_rag_agent/agent_executor.py`) wrapping a real agent, a `main.py` that builds an `AgentCard` + `DefaultRequestHandler` + `A2AStarletteApplication` and serves it with `uvicorn`, plus `utilities/a2a/agent_discovery.py` and `agent_connect.py` for other agents to discover and call it — just with a Google ADK agentic-RAG pipeline behind the executor instead of a LangGraph ReAct agent, and with a `host_agent` orchestrating multiple such A2A agents instead of one client cell. This notebook is the minimal, from-scratch teaching version of that same idea.

## Summary

**Key takeaways:**

- An A2A server is just a thin adapter: an `AgentExecutor.execute()` method that reads `RequestContext.get_user_input()`, drives *any* underlying agent (here, a LangGraph graph's `.invoke()`), and reports status/results back through a `TaskUpdater`.
- The `AgentCard` — served at the well-known `/.well-known/agent.json` path — is the contract: it advertises real skills (`calculator`, `lookup`) that map onto the agent's actual tools, so a caller knows what to ask for before ever sending a message.
- The framework used to build the agent (LangGraph here, Google ADK in `mcp_a2a_agentic_rag`, anything else elsewhere) is irrelevant to an A2A client — it only ever talks to the protocol surface (`A2ACardResolver`, `A2AClient`, `Task`/`Message`).
- This is notebook 2 of 3 in the A2A track: `01_Foundations/01_A2A_Protocol_Basics.ipynb` covers the raw protocol mechanics with a toy echo agent; `03_Applications/` (authored separately) will show cross-framework interop built on exactly this pattern.